In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("lab07.ipynb")

# Lab 7: Posterior Predictive Checks and Metropolis-Hastings for GLMs

Welcome to the seventh Data 102 lab!

The goal of this lab is to practice checking Bayesian models using posterior predictive checks, and to use the Metropolis-Hastings algorithm to fit a Bayesian logistic regression model under several different priors.

The code and responses you need to fill in are represented by `...`. There is additional documentation for each part as you go along.

## Collaboration Policy
Data science is a collaborative activity. While you may talk with others about the labs, we ask that you **write your solutions individually**.

## Completing the Lab

Labs are **not graded** this semester. They are provided as practice to help you engage with and reinforce the course material.

We strongly encourage you to work through the entire lab genuinely and use the provided tests to check your understanding. If you need help, you can attend lab-specific office hours on Mondays, tutoring, or other course office hours.

After completing the lab, we recommend navigating to **Kernel > Restart Kernel and Run All Cells** and ensuring that your code runs without errors. Restarting your kernel helps ensure that your code works from a clean state rather than relying on variables from previously executed cells.

Recommended completion by: **Wednesday, October 28th.**

`<Collaborator Name> <Collaborator e-mail>`

# Setup
Let's begin by importing the libraries we will use. You can find the documentation for the libraries here:
* matplotlib: https://matplotlib.org/3.1.1/contents.html
* numpy: https://docs.scipy.org/doc/
* pandas: https://pandas.pydata.org/pandas-docs/stable/
* seaborn: https://seaborn.pydata.org/
* PyMC: https://www.pymc.io/projects/docs/en/stable/
* ArviZ: https://python.arviz.org/en/stable/

In [ ]:
import numpy as np
import pandas as pd
pd.options.mode.chained_assignment = None

%matplotlib inline

import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import arviz as az

sns.set()

# Stop ArviZ from displaying (and closing) its own figures, so that the labels we add to its plots
# land on the same figure rather than on a new, empty one
az.rcParams["plot.matplotlib.show"] = False

# Question 1: Posterior Predictive Checks

We will compare a Poisson regression model that uses the sea-surface-temperature (SST) anomaly with a constant-rate model, using posterior predictive checks.

### Load the data

In [ ]:
# No need to modify: Just run the code to load the data
df = pd.read_csv("hurricane_data.csv")
df = df[["Year", "Num_Storms", "Temp_Anomaly"]]
df.tail()

### Model Specifications

The `Num_Storms` column contains the number of named storms recorded each year between 1880 and 2019. The `Temp_Anomaly` column contains the deviation in yearly SST from the mean of 1951-1980.

As in Lab 5, we will model the number of named storms in Year $i$ using **Poisson Regression**:

$$\lambda_i = e^{q_0 + q_1 X_i}$$

$$C_i \sim \text{Poisson}(\lambda_i),$$

where $X_i$ is the SST deviation in Year $i$, and $C_i$ is the number of named storms in year $i$.

## 1(a) Posterior Predictive Checks

In order to validate our Bayesian model we perform Posterior Predictive Checks (PPC). After performing Bayesian regression we have access to a generating distribution for counts $C'_i \mid X_i$. The crux of PPC is to sample such counts and to compare them to the original historical data.

The code below computes PPC samples and plots their distribution. Note that here, $C$ has been renamed $y$, and the band labeled "Posterior predictive $y$" is a collection of curves, each of which is the density plot of $y$ for a given draw of the coefficient vector $q = (q_0, q_1)$ from the posterior.

Specifically, to generate one of these curves we follow these steps: First, we take one of our posterior samples of $q_0$ and $q_1$. Then we draw samples from the distribution $p(Y_i \mid q_0, q_1, X_i)$ for each data point $X_i$. Then we plot the density of the resulting $Y$'s.

In [ ]:
# No need to modify
with pm.Model() as poisson_model:
    q0 = pm.Flat('q0')
    q1 = pm.Flat('q1')

    lam = np.exp(q0 + q1 * df['Temp_Anomaly'])

    y = pm.Poisson('y', mu=lam, observed=df['Num_Storms'])

    # DO NOT CHANGE THE SAMPLING ROUTINE
    trace_poisson = pm.sample(1000, cores=2, target_accept=0.95, init='adapt_diag', random_seed=102)

In [ ]:
# No need to modify
# Sample C'_i | X_i
with poisson_model:
    poisson_ppc = pm.sample_posterior_predictive(trace_poisson)
    poisson_ppc.posterior_predictive['y'] = poisson_ppc.posterior_predictive['y'] + 0.0
    poisson_ppc.observed_data['y'] = poisson_ppc.observed_data['y'] + 0.0

In [ ]:
# No need to modify: plot the PPC samples
ax = az.plot_ppc(poisson_ppc)
ax.set_xlabel('y = Number of named storms')
ax.set_title('Bayesian Poisson Regression with SST covariate')
ax.set_xlim(-2, 45)
ax.set_ylim(-0.01, 0.2);

To check how important the SST (our data $X_i$) is to the results above, we can estimate a new Bayesian model which doesn't use the covariates $X_i$, and instead uses a simple conjugate prior.

This simple model is implemented below.

In [ ]:
# No TODOs here: Just inspect the code and make sure you understand what is happening
with pm.Model() as simple_bayes_model:
    mu = pm.Gamma('mu', alpha=10, beta=1)
    counts = pm.Poisson('C', mu=mu, observed=df.Num_Storms)
    simple_trace_poisson = pm.sample(1000, cores=2, target_accept=0.95, init='adapt_diag', random_seed=102)

In [ ]:
# No need to modify
# Sample C'_i
with simple_bayes_model:
    simple_bayes_ppc = pm.sample_posterior_predictive(simple_trace_poisson)
    simple_bayes_ppc.posterior_predictive['C'] = simple_bayes_ppc.posterior_predictive['C'] + 0.0
    simple_bayes_ppc.observed_data['C'] = simple_bayes_ppc.observed_data['C'] + 0.0

In [ ]:
# No need to modify: plot the PPC samples
ax = az.plot_ppc(simple_bayes_ppc)
ax.set_xlabel('C = Number of named storms')
ax.set_title('Bayesian Poisson Rate Estimation without SST covariate')
ax.set_xlim(-2, 45)
ax.set_ylim(-0.01, 0.2);

<!-- BEGIN QUESTION -->

### 1(a)(i) With and without SST

Compare the two plots above. In your opinion, which model is a better fit for the observed data? Explain your answer in 2-3 sentences.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

# Question 2: Bayesian Logistic Regression

In the rest of this lab, you will explore how different priors regularize a Bayesian logistic regression, and you will use the Metropolis-Hastings algorithm to sample from the resulting posteriors. In a logistic regression, the likelihood is not conjugate to any standard prior family, so we cannot write down the posterior in closed form and must approximate it by sampling.

This question has four parts:

- **Part A:** derive and implement the likelihood, and implement the four priors.
- **Part B:** sample from the posterior using Metropolis-Hastings, and study how the proposal affects the sampler.
- **Part C:** compare the posteriors under four different priors.
- **Part D:** discuss what you observed.

## Part A: Likelihood and Priors

### Data

The data in this question are drawn from:
- Racine et al. *Bayesian Methods in Practice: Experiences in the Pharmaceutical Industry.* Journal of the Royal Statistical Society. 1986. [JSTOR link](https://www.jstor.org/stable/2347264) (this is the source for the toxicity data set used in Chapter 3.7 of *Bayesian Data Analysis* (BDA) by Gelman et al.)

The article introduces a Bayesian regression model for the toxicity of a substance based on an animal trial. Establishing the toxicity of chemical substances is important in pharmaceutical research, but animal trials that harm their subjects should only be performed when necessary, and on as few animals as possible. 

Since these studies were, as of Racine's writing, performed routinely, past information (or information from studies on similar substances) could be used to inform priors. The need for robust analyses from small sample sizes motivates the use of Bayesian analysis here. For further reasons to adopt a Bayesian approach, see [Racine et al](https://www.jstor.org/stable/2347264).

The data consist of four trials for the acute toxicity of a substance, each involving five subject animals (20 animals in total). The dose $\{x_i\}_{i=1}^4$ varied between the four groups, increasing from 422 mg/ml to 2069 mg/ml. In each group, the number of animals killed by the substance, $\{y_i\}_{i=1}^4$, was recorded.

In [ ]:
# No need to modify: Just run the code to load the data
toxicity_data = pd.read_csv("toxicology_data.csv")
toxicity_data

### Model

Following BDA, consider the logistic regression model:

$$
\begin{aligned}
\text{Linear combination of features:} \quad & \eta(x,\beta) = \beta_0 + \beta_1 \log(x) \\
\text{Logistic (inverse) link:} \quad & q(x,\beta) = \text{logistic}(\eta(x,\beta)) = \left[1 + \exp(-\eta(x,\beta))\right]^{-1} \\
\text{Binomial sampling distribution:} \quad & Y \mid x,\beta \sim \text{Bin}\left(n, q(x,\beta)\right)
\end{aligned}
$$

where $n$ is the number of animals exposed to each dose (here $n = 5$ for every trial).

Here $\eta$ is the *linear predictor*. The logistic function is the inverse of the logit (log-odds) link, so $\eta(x,\beta) = \log\frac{q}{1-q}$ is the log-odds that an animal exposed to dose $x$ is harmed. This is the same logistic regression as in the GLM lecture and the [GLM chapter of the textbook](https://data102.org/ds-102-book/content/chapters/03/glms/), except that the response is **Binomial** instead of Bernoulli: $Y$ counts how many of the $n$ animals exposed to a dose are harmed, which is the sum of $n$ Bernoulli outcomes with the same probability $q$.

### Inferential Goal

We want to infer the regression coefficients $\beta = [\beta_0,\beta_1]$ from the data. In particular, toxicity regulation is based on the dosage level $x$ at which the substance has a 50% chance of causing harm (here defined as causing a fatal reaction), called the **50% lethal dose** (LD50). This is the level at which $\eta(x,\beta) = 0$, that is,

$$x_*(\beta) = \exp(-\beta_0/\beta_1).$$

We will study how the posterior on $x_*$ changes under different prior specifications.

<!-- BEGIN QUESTION -->

### 2(a) Derive the likelihood, then the log likelihood

Given the data $\{(x_i,y_i)\}_{i=1}^4$, show that the likelihood of the regression parameters $\beta$ is, up to proportionality:

$$
\text{Lik}(\beta) \propto  \prod_{i=1}^4 \left(1 + e^{-\beta_0 - \beta_1 \log(x_i)}\right)^{-y_i} \left(1 + e^{\beta_0 + \beta_1 \log(x_i)}\right)^{-(n-y_i)}
$$

Fully justify each term in this equation.

_Type your answer here, replacing this text._

Derive the log likelihood $\log \text{Lik}(\beta)$, **up to additive constants that do not depend on $\beta$** (the sampler we use later only ever compares the posterior at two values of $\beta$, so any term that does not depend on $\beta$ cancels and can be dropped), as a sum over the four trials. Write your answer in terms of $\eta_i = \beta_0 + \beta_1 \log(x_i)$, $y_i$, and $n$, and show your steps (start from the log of the product of binomial likelihood terms, then substitute the logistic link).

_Type your answer here, replacing this text._

<!-- END QUESTION -->

### 2(b) Implement the log likelihood

Fill in the `trial_log_lik = ...` line below with the expression you derived in 2(a) for a single trial. The function loops over the four trials, so inside the loop `x`, `y` and `n` are the dose, the number of animals harmed, and the number of animals exposed for that trial, and `eta` is $\eta_i$. Each trial's contribution is then added to `log_prob`.

*Hint:* `np.log` and `np.exp` are the natural logarithm and exponential. For very large `eta`, `np.exp(eta)` can overflow, but this will not matter for the tests below.

In [ ]:
def log_likelihood_toxicity(beta, data):
    # Returns the log likelihood of beta = [beta_0, beta_1], up to an additive constant
    log_prob = 0
    for x, y, n in zip(data['Dose_(mg/ml)'], data['Number_Harmed'], data['Number_Exposed']):
        eta = beta[0] + beta[1] * np.log(x)

        trial_log_lik = ...
        log_prob += trial_log_lik
    return log_prob

In [ ]:
grader.check("q2b")

### Priors

We will consider four possible choices of prior on $\beta$. They are listed below in order of generality: each includes its predecessors as a special case.

1. **Uniform.** An *improper* uniform prior on $\beta$ (see both BDA and Racine). Improper means that the prior density is constant over all of $\mathbb{R}^2$, so it cannot be normalized to integrate to 1. We can still use it in Bayes' rule, since we only need the prior up to a constant, but the resulting posterior is only guaranteed to be a proper distribution if the likelihood decays fast enough. We will see that, for this data set, it does not.

2. **Expert.** A Gaussian prior on $\beta$ formed from consultation with toxicology experts (see Racine):

   $$
   \beta \sim \mathcal{N}(\bar{\beta}, C_{\beta}(\sigma)), \qquad \bar{\beta} = \begin{bmatrix} -17.31 \\ 2.57 \end{bmatrix}, \qquad C_{\beta}(\sigma) = \begin{bmatrix} 1053.72 & -156.45 \\ -156.45 & 23.24 \end{bmatrix} + \sigma^2 I,
   $$

   where $\sigma > 0$ is a free parameter introduced to weaken the prior. When $\sigma = 0$, this prior is *singular*: the covariance matrix is not invertible, and all of the prior density lies on a line.

3. **Regularized.** A Gaussian prior on $\beta$ designed to introduce a stronger regularizing effect. In particular, use the covariance

   $$
   R_{\beta}(\sigma,\mu) = \left(C_{\beta}(\sigma)^{-1} + \mu^{-2} I\right)^{-1}.
   $$

   Adding $\mu^{-2} I$ inside the inverse shrinks the prior covariance: the prior variance is smaller than under $C_{\beta}$ in every direction, and in no direction can it exceed $\mu^2$. The smaller $\mu$, the tighter the prior and the larger the regularizing effect. (*Regularization* is the same idea as in the linear-model lectures: the prior pulls the estimate towards $\bar{\beta}$, trading a little bias for less variance.)

4. **Robust.** A yet weaker prior formed by replacing the Gaussian density with a Student-$t$ density

   $$
   p_{\beta}(\beta) \propto \left(1 + \frac{1}{\nu} (\beta - \bar{\beta})^{\intercal} R_{\beta}(\sigma,\mu)^{-1} (\beta - \bar{\beta}) \right)^{-\frac{\nu + 2}{2}},
   $$

   where $\nu$ is a shape parameter (the *degrees of freedom* of the Student-$t$ distribution). The Student-$t$ density has *heavy tails*: while the Gaussian density decays faster than exponentially away from its center, the $t$ density decays only as a power of the distance, so values far from $\bar{\beta}$ remain plausible under the prior. When $\nu = 1$ the prior is a multivariate Cauchy. As $\nu \rightarrow \infty$, the prior converges to the multivariate normal prior with mean $\bar{\beta}$ and covariance $R_{\beta}$.

Notice that the 4th prior family encompasses the previous three, so it is enough to implement (4) and then pick the parameters appropriately:

- The **uniform** prior is recovered by setting $\sigma$ and $\mu$ arbitrarily large and $\nu$ small.
- The **expert** prior is recovered by setting $\mu$ and $\nu$ arbitrarily large.
- The **regularized** prior is recovered by setting $\nu$ arbitrarily large.

In general, the prior parameters $\sigma, \mu, \nu$ may be interpreted as follows:

1. $\sigma$ weakens the expert-provided prior by adding variance. When $\sigma = 0$ the expert-provided prior assumes that $\beta_0$ and $\beta_1$ satisfy a strict linear relationship. Using a (small) nonzero $\sigma$ allows some variation about this relationship.
2. $\mu$ tightens the prior by introducing a soft threshold on the prior variance: the smaller $\mu$, the smaller the allowed variance. This is an important regularization parameter, since the expert-provided prior allows a large degree of variation in the regression parameters along a direction in parameter space that is not well constrained by the likelihood.
3. $\nu$ relaxes the tails of the prior so that they decay according to a power law. This fat-tailed behavior allows more outliers when sampling from the prior, so it does not as strongly suppress the tails of the posterior. This makes inference more robust: the slower the prior tails decay, the less bias the prior introduces when the data suggest that the true parameters would have been surprising under the prior. The posterior is then more reactive to the likelihood, which increases its variance while decreasing the bias introduced by regularization.

In the next four questions you will implement the log prior density of each of these priors, **up to an additive constant**. Each function returns the log prior density for a given $\beta$; we will use them in the sampler below.

### 2(c) Implement the uniform prior

Fill in the function below, which returns the log density of the uniform prior (up to an additive constant).

In [ ]:
# Implement the uniform prior. Your code should return the log prior density up to an additive constant
def log_prior_uni_toxicity(beta):
    log_prob = ...
    return log_prob

In [ ]:
grader.check("q2c")

### 2(d) Implement the expert (normal) prior

Fill in the function below, which returns the log density of the expert prior $\beta \sim \mathcal{N}(\bar{\beta}, C_\beta(\sigma))$ (up to an additive constant). `beta_centered` is $\beta - \bar{\beta}$.

In [ ]:
# Implement the normal prior. Your code should return the log prior density up to an additive constant
def log_prior_normal_toxicity(beta, sigma):
    beta_centered = beta - np.array([-17.31, 2.57])
    C = ...
    log_prob = ...
    return log_prob

In [ ]:
grader.check("q2d")

### 2(e) Implement the regularized normal prior

Fill in the function below, which returns the log density of the regularized normal prior $\beta \sim \mathcal{N}(\bar{\beta}, R_\beta(\sigma, \mu))$ (up to an additive constant), where $R_\beta(\sigma,\mu) = \left(C_\beta(\sigma)^{-1} + \mu^{-2} I\right)^{-1}$.

In [ ]:
# Implement the regularized normal prior. Your code should return the log prior density up to an additive constant
def log_prior_reg_normal_toxicity(beta, sigma, mu):
    beta_centered = beta - np.array([-17.31, 2.57])
    C = np.array([[1053.72, -156.45], [-156.45, 23.24]]) + (sigma**2) * np.eye(2)
    R = ...
    log_prob = ...
    return log_prob

In [ ]:
grader.check("q2e")

### 2(f) Implement the robust (Student-$t$) prior

Fill in the function below, which returns the log density of the Student-$t$ prior (up to an additive constant): $\log p_\beta(\beta) = -\frac{\nu + 2}{2} \log\left(1 + \frac{1}{\nu}(\beta - \bar{\beta})^{\intercal} R_\beta^{-1} (\beta - \bar{\beta})\right) + \text{const}$.

In [ ]:
# Implement the student-t prior. Your code should return the log prior density up to an additive constant
def log_prior_student_toxicity(beta, sigma, mu, nu):
    beta_centered = beta - np.array([-17.31, 2.57])
    C = np.array([[1053.72, -156.45], [-156.45, 23.24]]) + (sigma**2) * np.eye(2)
    R = np.linalg.inv(np.linalg.inv(C) + mu**(-2) * np.eye(2))
    quadratic_term = ...
    log_prob = ...
    return log_prob

In [ ]:
grader.check("q2f")

## Part B: Metropolis-Hastings Sampling

### The sampler

In Lab 6 you used PyMC and built a Gibbs sampler, and the [sampling section of the textbook](https://data102.org/ds-102-book/content/chapters/02/inference-with-sampling/) describes rejection sampling, which draws independent proposals and throws some of them away. Both are ways of approximating a posterior with samples. Here we use a third method, **Metropolis-Hastings**, a Markov chain Monte Carlo (MCMC) method like Gibbs sampling. It builds a sequence of samples in which each sample depends on the previous one, in such a way that the chain's long-run (steady-state) distribution is the posterior. Unlike Gibbs sampling, it does not require us to be able to sample from the conditional distributions of the posterior, and it only needs the posterior up to a constant. (The textbook only sketches this algorithm in an optional section, so we describe it in full here. For details, see the slides from [lecture 14](https://data102.org/fa26/lectures/sampling-2/).)

At each iteration, starting from the current value $\beta$:

1. **Propose** a new value $\beta' = \beta + \text{step\_size} \cdot \epsilon$, where $\epsilon$ is Gaussian noise with independent entries whose standard deviations are given by `scales`. This is a symmetric proposal.
2. **Accept** the proposal with probability $\min\left(1, \dfrac{p(\beta' \mid \text{data})}{p(\beta \mid \text{data})}\right)$. This ratio of posterior densities (the *posterior odds* of the proposal against the current value) only requires the *unnormalized* posterior, that is, the likelihood times the prior, because the normalizing constant cancels. A proposal that is more probable than the current value is always accepted, and a less probable one is accepted with probability equal to the ratio. Otherwise, **reject** the proposal and stay at $\beta$. 
3. Record the current value of $\beta$. The chain starts at an arbitrary point, so samples before `burn_in` are discarded while the chain moves to the region where the posterior has mass. Only every `stride`-th sample is kept (this is called *thinning*), to reduce the correlation between neighboring samples.

You will not need to modify this function, but you will experiment with its tuning parameters `step_size` and `scales`. The function also returns the **acceptance rate**: the fraction of proposals that were accepted. You should read it carefully to make sure you understand the Metropolis-Hastings implementation. Pseudo-code for MH samplers will be in scope for exam 2.

In [ ]:
def metropolis_hastings(beta, data, prior_param, scales=[7, 1], iterations=200000, step_size=0.1, burn_in=1000, stride=100):
    # initialize chain
    samples = []
    n_accepted = 0

    sigma, mu, nu = prior_param
    current_ll = log_likelihood_toxicity(beta, data)
    current_lpr = log_prior_student_toxicity(beta, sigma, mu, nu)
    current_lpo = current_ll + current_lpr

    # iterate the Markov chain
    for k in range(1, iterations + 1):
        # propose new sample location
        new_beta = beta + step_size * np.random.normal(0, scales, size=2)

        # evaluate proposal
        new_ll = log_likelihood_toxicity(new_beta, data)
        new_lpr = log_prior_student_toxicity(new_beta, sigma, mu, nu)
        new_lpo = new_ll + new_lpr

        # decide whether to accept or reject the proposal
        acceptance_ratio = np.exp(min(new_lpo - current_lpo, 0)) # posterior odds, capped at 1
        if np.random.rand() < acceptance_ratio:
            beta = new_beta
            current_lpo = new_lpo
            n_accepted += 1

        # add to the running list of samples
        if k % stride == 1 and k >= burn_in:
            samples.append(beta.copy())
    return np.array(samples), n_accepted / iterations

### Inspecting a run

It is good practice to inspect the sample runs, using *trace plots* (the sampled value of each parameter plotted against the iteration number). For a chain that has mixed over its stationary distribution, you should see a stochastic process where:

- the entries are not overly correlated with each other (the process should look noisy and not overly smooth);
- there is no general trend in any direction; and
- the empirical distribution of values on any smaller interval selected at random is close to the empirical distribution over the full sample. In other words, you should not be able to change the distribution of samples significantly by truncating, or zooming in on, an interval of roughly 1/5th the length of the full sample.

In short, a well-mixed chain should look like a stationary noise process.

A few terms that describe these properties: a chain *mixes* well if it moves quickly around the whole posterior, and its *stationary distribution* is the posterior that it eventually samples from. Neighboring samples in a poorly mixing chain are highly correlated (they have high *autocorrelation*), so such a chain carries much less information than the same number of independent samples. The *effective sample size* is the number of independent samples that would estimate a posterior summary about as accurately as the chain does: it is much smaller than the number of retained samples when the autocorrelation is high.

We first set the prior parameters. These values correspond to a reasonably weak prior: close to the expert-provided prior ($\sigma = 0.1$), with weak regularization ($\mu = 10$) and close to Gaussian tails ($\nu = 40$). We will use them as our **reference** prior.

In [ ]:
# Set the prior parameters
sigma = 0.1 # reasonably close to the expert-provided prior
mu = 10     # weak regularization
nu = 40     # close to Gaussian tails
prior_param = np.array([sigma, mu, nu])

# Initialize to the expert prior expectation
beta_init = np.array([-17.31, 2.57])

# Helper function: plot the trace of each coefficient for a list of runs
def plot_traces(runs, labels):
    fig, axs = plt.subplots(2, len(runs), figsize=(5 * len(runs), 6), squeeze=False)
    for j, (run, label) in enumerate(zip(runs, labels)):
        axs[0, j].plot(run[:, 0])
        axs[0, j].set_title(label)
        axs[0, j].set_ylabel("Intercept")
        axs[1, j].plot(run[:, 1])
        axs[1, j].set_ylabel("Slope")
        axs[1, j].set_xlabel("Retained sample")
    plt.tight_layout()
    plt.show()

### Proposal scales

The proposal in the sampler perturbs each coefficient by independent Gaussian noise: the jump in coefficient $j$ has standard deviation `step_size * scales[j]`. The `scales` argument sets the relative size of the perturbation for the intercept $\beta_0$ and the slope $\beta_1$, and the default is `scales=[7, 1]`. With the default `step_size=0.1`, a typical jump therefore has size about $0.7$ in the intercept and about $0.1$ in the slope.

We will compare the default proposal with an **isotropic** proposal, `scales=[1, 1]`, that perturbs both coefficients equally (*isotropic* means that the proposal is equally spread in every direction). Run the cell below to run three chains, one after another. These runs may take a minute or so each, so let them simmer.

- **Chain A:** `scales=[1, 1]` with `step_size=0.7`. The intercept jumps have the same size (about $0.7$) as under the default proposal.
- **Chain B:** `scales=[1, 1]` with `step_size=0.1`. This is what we get if we respond to the rejections in Chain A by shortening the step.
- **Chain C:** `scales=[7, 1]` (the default) with `step_size=0.1`.

In [ ]:
# Chain A: scales [1, 1], large step size
np.random.seed(102)
samples_A, accept_A = metropolis_hastings(beta_init, toxicity_data, prior_param, scales=[1, 1], step_size=0.7)

# Chain B: scales [1, 1], short step size
np.random.seed(102)
samples_B, accept_B = metropolis_hastings(beta_init, toxicity_data, prior_param, scales=[1, 1], step_size=0.1)

# Chain C: scales [7, 1] (the default), step size 0.1
np.random.seed(102)
post_samples, accept_C = metropolis_hastings(beta_init, toxicity_data, prior_param, scales=[7, 1], step_size=0.1)

print("Chain A: scales [1, 1], step_size 0.7, acceptance rate {:.2f}".format(accept_A))
print("Chain B: scales [1, 1], step_size 0.1, acceptance rate {:.2f}".format(accept_B))
print("Chain C: scales [7, 1], step_size 0.1, acceptance rate {:.2f}".format(accept_C))

plot_traces([samples_A, samples_B, post_samples],
            ["A: scales [1, 1], step_size 0.7", "B: scales [1, 1], step_size 0.1", "C: scales [7, 1], step_size 0.1"])

<!-- BEGIN QUESTION -->

### 2(g) Rejected proposals

Compare the acceptance rates of Chain A and Chain C, whose intercept jumps have the same size. What fraction of the proposals does the isotropic proposal reject, compared to the default proposal? Chain B shortens the step size to fix this: what happens to the acceptance rate? Answer in 2-3 sentences.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### 2(h) Which chain converged?

Using the trace plots and the criteria described above, which of the three chains appear to have mixed over the posterior distribution? Describe what you see in each trace plot. If you had a fixed budget of iterations, which chain would you rather use, and why? Answer in 3-4 sentences.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

### Posterior summary

The code below uses the (well-mixed) `scales=[7, 1]` run, `post_samples`, to return approximations to the:
1. **posterior mean** estimator for each entry of $\beta$,
2. **95% credible intervals** for each entry of $\beta$ (a 95% credible interval contains 95% of the posterior probability. We compute it from the samples as the 2.5th to 97.5th percentiles),
3. **posterior mean** and **95% credible interval** for $x_*(\beta)$, the dose at which the substance has a 50% chance of causing a fatal reaction.

We define helper functions here because we will reuse them for each prior below.

In [ ]:
# Helper function: print posterior summaries for beta and x_*, and return the posterior samples of x_*
def summarize_posterior(samples):
    betas_posterior_means = samples.mean(axis=0)
    betas_credible_intervals = np.percentile(samples, [2.5, 97.5], axis=0)

    print("The posterior mean for the intercept is: {}".format(np.round(betas_posterior_means[0], 1)))
    print("with 95 percent credible interval: {}".format(np.round(betas_credible_intervals[:, 0], 1)))
    print("")
    print("The posterior mean for the slope is: {}".format(np.round(betas_posterior_means[1], 1)))
    print("with 95 percent credible interval: {}".format(np.round(betas_credible_intervals[:, 1], 1)))
    print("")

    x_stars = np.exp(-samples[:, 0] / samples[:, 1])
    print("The posterior mean for the 50 percent lethality dose is: {}".format(np.round(x_stars.mean(), 1)))
    print("with 95 percent credible interval: {}".format(np.round(np.percentile(x_stars, [2.5, 97.5]), 1)))
    return x_stars

x_stars = summarize_posterior(post_samples)

#### Posterior visualization

First, let's look at the resulting posterior distribution on the regression coefficients $\beta$. This is approximated below using the posterior samples drawn above.

In [ ]:
plt.scatter(post_samples[:, 1], post_samples[:, 0], s=0.2)
plt.ylim([-60, 0])
plt.xlim([0, 8.5])
plt.xlabel("Slope")
plt.ylabel("Intercept")
plt.show()

Next, let's visualize the posterior for $\log(x_*(\beta))$, the log concentration at which the compound is expected to kill 50% of subjects exposed to it. For consistency with the analysis in Racine, this posterior should put most of its mass between 6.5 and 7, with a peak near 6.75.

In [ ]:
plt.hist(np.log(x_stars), 70, density=True, edgecolor='none')
plt.xlim([6, 8])
plt.xlabel("Log 50 percent lethality dose")
plt.show()

Finally, let's quantify the shape of the posterior in $\beta$. The cell below prints the posterior standard deviation of each coefficient and the correlation between them, from the samples in `post_samples`.

In [ ]:
print("Posterior standard deviations (intercept, slope):", np.round(post_samples.std(axis=0), 2))
print("Ratio of the intercept standard deviation to the slope standard deviation:", np.round(post_samples[:, 0].std() / post_samples[:, 1].std(), 1))
print("Posterior correlation between intercept and slope:", np.round(np.corrcoef(post_samples.T)[0, 1], 3))

<!-- BEGIN QUESTION -->

### 2(i) Why does `scales=[7, 1]` mix more efficiently?

Use the scatter plot and the summary statistics above to explain why MCMC with the proposal scales `[7, 1]` mixes more efficiently than MCMC with scales `[1, 1]`. In your answer, describe the shape of the posterior in $(\beta_0, \beta_1)$, and what it implies about how each proposal needs to be shaped. Answer in 3-5 sentences.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

## Part C: Comparing Priors

Now, we compare the posterior under the reference prior with the posteriors under the four prior specifications described above. For each specification below, rerun the sampling and posterior visualization code. Inspect the change in each posterior summary, and think about how the change in the prior changes the posterior. We have defined a helper function that runs the sampler with the default `scales=[7, 1]`, prints posterior summaries, and plots the new posterior samples over the reference samples.

There are no questions in this part: run the cells and look at the output carefully, since you will need your observations for the discussion questions in Part D.

In [ ]:
# Helper function: sample from the posterior under a new prior, then compare to the reference run
def run_prior_experiment(sigma, mu, nu, label, ylim, xlim):
    np.random.seed(102)
    new_samples, new_accept = metropolis_hastings(beta_init, toxicity_data, np.array([sigma, mu, nu]))
    print("Acceptance rate: {:.2f}\n".format(new_accept))
    new_x_stars = summarize_posterior(new_samples)

    plt.scatter(post_samples[:, 1], post_samples[:, 0], s=0.2)
    plt.scatter(new_samples[:, 1], new_samples[:, 0], s=0.2)
    plt.ylim(ylim)
    plt.xlim(xlim)
    plt.xlabel("Slope")
    plt.ylabel("Intercept")
    plt.legend(['Reference', label])
    plt.show()

    plt.hist(np.log(x_stars), 70, alpha=0.8, density=True, edgecolor='none')
    plt.hist(np.log(new_x_stars), 70, alpha=0.8, density=True, edgecolor='none')
    plt.xlim([6, 8])
    plt.xlabel("Log 50 percent lethality dose")
    plt.legend(['Reference', label])
    plt.show()
    return new_samples

### Uniform

Start with a near-uniform prior. This can be achieved by setting $\sigma$ very large, $\mu$ very large, and $\nu$ very small. Use $\sigma = 400$, $\mu = 1000$ and $\nu = 1$. This is a very flat prior. You should notice that, since the data are "well separated" (the fraction of animals harmed rises from 0/5 at the lowest dose to 5/5 at the highest, so a very steep logistic curve fits almost as well as a moderate one), the likelihood alone places essentially no upper bound on the unknown parameters. This is a case where the MLE would be highly sensitive to changes in the data, and may be unbounded above.

In [ ]:
uniform_samples = run_prior_experiment(sigma=400, mu=1e3, nu=1, label='Uniform', ylim=[-120, 0], xlim=[0, 17])

### Expert

Now, adopt the expert prior. Set $\sigma = 0.1$, $\mu = 1000$, and $\nu = 40$. Does this succeed in constraining the unknown parameters? Pay attention to the axis scales when you read the plots below: we have zoomed in.

In [ ]:
expert_samples = run_prior_experiment(sigma=0.1, mu=1e3, nu=40, label='Expert', ylim=[-100, 0], xlim=[0, 15])

### Regularized

Regularize by restricting the variance. Leave $\sigma = 0.1$ and $\nu = 40$, and try setting $\mu = 5$, then $\mu = 1$, and then $\mu = 0.2$. (Change `mu` in the cell below and rerun it for each value.)

In [ ]:
mu = 5 # try 5, then 1, then 0.2   << CHANGE ME
regularized_samples = run_prior_experiment(sigma=0.1, mu=mu, nu=40, label='Regularized', ylim=[-60, 0], xlim=[0, 10])

### Robust

Set $\sigma = 0.1$ and $\mu = 5$. Relax the tails of the prior by setting $\nu = 4$, then $\nu = 2$, and then $\nu = 1$. (Change `nu` in the cell below and rerun it for each value.) Pay attention to the outlying samples.

In [ ]:
nu = 2 # try 4, then 2, then 1   << CHANGE ME
robust_samples = run_prior_experiment(sigma=0.1, mu=5, nu=nu, label='Robust', ylim=[-60, 0], xlim=[0, 10])

## Part D: Discussion

Use your observations from Part C to answer the following conceptual questions.

<!-- BEGIN QUESTION -->

### 2(j) Uniform prior

Under the near-uniform prior, the posterior samples for the slope and the intercept spread over a much larger range than under the reference prior. What does this tell you about how much information the *likelihood alone* provides about $\beta$ for this data set? Why does this happen here (think about the four observed proportions harmed), and what does it suggest about maximum likelihood estimation? Answer in 2-4 sentences.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### 2(k) Regularization

As you decrease $\mu$ (from 5 to 1 to 0.2), what happens to the posterior for $\beta$ and to the 95% credible interval for the 50% lethal dose $x_*$? Does the prior make the estimate of $x_*$ biased? Describe the bias-variance tradeoff in 2-4 sentences.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### 2(l) Robust prior

As you decrease $\nu$ (from 4 to 2 to 1), how do the posterior samples change, and why? Think about the tails of the prior. In what situation would you prefer a heavy-tailed prior to a Gaussian prior with the same scale? Answer in 2-4 sentences.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### 2(m) Choosing a prior for regulation

Suppose you must infer the 50% lethal dose for use in writing regulation. Which prior (or set of priors) would you choose, and how would you justify it to a regulator? In your answer, consider:

- the degree to which it balances induced bias for reduced variance (in the posterior, and in sampling over new data sets);
- whether you can anticipate the types of errors it would induce if (as is certain) it is misspecified;
- whether you can justify the errors induced by misspecification as errors in the "safe" direction (consider the differential impacts of underestimating versus overestimating the 50% lethal dose); and
- how you would distinguish *within-model* uncertainty (posterior variance) from *modeling* uncertainty (the sensitivity of the conclusions to the choice of prior) when reporting your conclusions.

Answer in 4-8 sentences.

_Type your answer here, replacing this text._

<!-- END QUESTION -->



---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit.

In [ ]:
grader.export(pdf=False, force_save=True)